# Jung et al. 2001 Citation Metadata

Source paper: Jung et al. 2001, DOI `10.1002/hbm.1050`, OpenAlex work `W1990928820`. This notebook is for citation metadata only; it is not an EEG/ERP raw-data import.

In [ ]:
import Pkg

function find_repo_root()
    candidates = unique(normpath.([
        pwd(),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
        joinpath(pwd(), "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "week_19", "data_sources")
const DATASETS_ROOT = joinpath(REPO_ROOT, "notebooks", "datasets")
const WEEK19_DOWNLOADS = joinpath(REPO_ROOT, "notebooks", "week_19", "downloads")
const PYTHON = begin
    venv_python = joinpath(REPO_ROOT, ".venv_8bit", "bin", "python")
    isfile(venv_python) ? venv_python : "python"
end

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))

using CairoMakie
using CSV
using DataFrames
using HDF5
using JSON3
using Printf
using Statistics

include(joinpath(REPO_ROOT, "notebooks", "week_15", "try_new_data_helpers.jl"))
using .Week15TryNewData

mkpath(WEEK19_DOWNLOADS)
println("Repo root: ", REPO_ROOT)
println("Python: ", PYTHON)

using Downloads

In [ ]:
const JUNG_OPENALEX_WORK_ID = "W1990928820"
const JUNG_DOI = "10.1002/hbm.1050"
const OPENALEX_DIR = joinpath(WEEK19_DOWNLOADS, "openalex")
const JUNG_CITATIONS_CSV = joinpath(OPENALEX_DIR, "jung_hbm_2001_citations_openalex.csv")
const RUN_IMPORT = false

function download_openalex_citations!()
    mkpath(OPENALEX_DIR)
    rows = NamedTuple[]
    for page in 1:20
        url = "https://api.openalex.org/works?filter=cites:$(JUNG_OPENALEX_WORK_ID)&per-page=200&page=$(page)&sort=publication_year:desc&select=id,doi,display_name,publication_year,cited_by_count,authorships,primary_location,open_access,type"
        json_path = joinpath(OPENALEX_DIR, "jung_hbm_2001_citations_page_$(page).json")
        Downloads.download(url, json_path)
        data = JSON3.read(read(json_path, String))
        for work in data.results
            loc = :primary_location in propertynames(work) ? work.primary_location : nothing
            source = loc === nothing || !(:source in propertynames(loc)) ? nothing : loc.source
            push!(rows, (
                openalex_id = String(work.id),
                doi = :doi in propertynames(work) && work.doi !== nothing ? String(work.doi) : "",
                title = String(work.display_name),
                publication_year = Int(work.publication_year),
                cited_by_count = Int(work.cited_by_count),
                source = source === nothing || source === nothing ? "" : String(source.display_name),
                landing_page_url = loc === nothing || !(:landing_page_url in propertynames(loc)) || loc.landing_page_url === nothing ? "" : String(loc.landing_page_url),
            ))
        end
        length(data.results) < 200 && break
        sleep(0.2)
    end
    CSV.write(JUNG_CITATIONS_CSV, DataFrame(rows))
end

if isfile(JUNG_CITATIONS_CSV)
    println("Using cached citation CSV: ", JUNG_CITATIONS_CSV)
elseif RUN_IMPORT
    download_openalex_citations!()
else
    @info "RUN_IMPORT=false; citation CSV missing. Set RUN_IMPORT=true to download."
end

citation_df = isfile(JUNG_CITATIONS_CSV) ? CSV.read(JUNG_CITATIONS_CSV, DataFrame) : DataFrame()
println("Citation rows: ", nrow(citation_df))
first(sort(citation_df, :cited_by_count, rev = true), min(20, nrow(citation_df)))

In [ ]:
if nrow(citation_df) > 0
    citation_by_year = combine(groupby(dropmissing(citation_df, :publication_year), :publication_year), nrow => :n_citing_papers)
    sort!(citation_by_year, :publication_year)
    fig = Figure(size = (980, 420), figure_padding = 18)
    ax = Axis(fig[1, 1]; title = "OpenAlex papers citing Jung et al. 2001", xlabel = "publication year", ylabel = "citing papers")
    barplot!(ax, citation_by_year.publication_year, citation_by_year.n_citing_papers; color = :gray35)
    fig
else
    println("No citation data loaded.")
end